# Step 4 - BL depth and final map dataset

Computes the BL depth from the ion spectra (distance to the neighbouring MSP and MSH spectra, `compute_depth`), its global rank-uniformization (`uniform_depthonly_spectro`, used by the maps) and its local one, the dipole tilt, and saves the 28 columns used by the maps.

Input: output of step 3. Output: `{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl`.

DepthComputation cells 4, 6, 7, 10 were dropped: they only add columns (logs, normalised features) that are not in the final file.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # makes the blxline package importable
from blxline.paths import OUT, MODEL_LINES  # all outputs of this repo are written under OUT (never over the original files)

In [ ]:
# [from ReorderingTemporalData/DepthComputation.ipynb, cell 0]
import matplotlib.pyplot as plt
import pandas as pd
from blxline.model import choose_best_features
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from spok.models.planetary import mp_shue1998_tangents, mp_shue1998_normal
from sklearn.preprocessing import StandardScaler
from blxline.maps import make_slice, gaussian_filter_nan_datas, reposition
from scipy.stats import binned_statistic_2d
from spok.models.planetary import get_tilt
from spok.models.planetary import mp_shue1997


In [ ]:
# [from ReorderingTemporalData/DepthComputation.ipynb, cell 11]
# Choose weights? For now, let's give them all the same weight

"""
if log:
    path = f'/DATA/ghisalberti/Datasets/BL_MMS1_THEMIS_Cluster_for_depth_log_spectro_07-01-2026.pkl'
else:
    path = f'/DATA/ghisalberti/Datasets/BL_MMS1_THEMIS_Cluster_for_depth_07-01-2026.pkl'
BL = pd.read_pickle(path)
"""

def get_features_and_weights(**kwargs):
    normalize = kwargs.get('normalize',False)
    log = kwargs.get('log',False)
    all_features = choose_best_features()
    all_features_unique = []
    for features in all_features:
        all_features_unique += list(features)
    all_features_unique = np.unique(np.array(all_features_unique).flatten())
    
    features_for_distance = []
    for feat in all_features_unique:
        #if feat not in ['energy_main_pop','flux_main_pop','width_main_pop','Np','Tp','Tpara','Tperp','Vn_MP','Bn_MP']:
        if feat not in ['center_coldions', 'max_coldions', 'std_coldions','energy_main_pop','flux_main_pop','width_main_pop','Np','Tp','Tpara','Tperp','Vn_MP','Vtan1_MP','Vtan2_MP','Vx','Vy','Vz','V','Bn_MP']:
            if 'ref_MSH' not in str(feat) and 'gap_to_MSH' not in str(feat):
                if 'spectro' in str(feat):
                    if log:
                        features_for_distance += ['log'+feat]
                    else:
                        features_for_distance += [feat]      
                else:
                    features_for_distance += [feat]
    
    if normalize:
        features_for_distance = ['normalized_'+col for col in features_for_distance]
        
    weights = [1/len(features_for_distance) for i in range(len(features_for_distance))]
    return features_for_distance, weights


def compute_depth(BL, features_for_distance, weights, description):
    # Compute dist to ref MSP
    dist = np.zeros(len(BL))
    for i, feat in enumerate(features_for_distance):
        dist += ((BL[feat].values - BL[feat+'_MSP'].values)**2).astype('float')*weights[i]
    dist = np.sqrt(dist)
    
    # Compute dist to ref MSH
    dist_to_msh = np.zeros(len(BL))
    for i, feat in enumerate(features_for_distance):
        dist_to_msh += ((BL[feat].values - BL[feat+'_MSH'].values)**2).astype('float')*weights[i]
    dist_to_msh = np.sqrt(dist_to_msh)
    
    # Compute dist between ref MSP and ref MSH
    dist_msh_to_msp = np.zeros(len(BL))
    for i, feat in enumerate(features_for_distance):
        dist_msh_to_msp += ((BL[feat+'_MSH'].values - BL[feat+'_MSP'].values)**2).astype('float')*weights[i]
    dist_msh_to_msp = np.sqrt(dist_msh_to_msp)
    
    
    depth = 1/2*(1+(dist**2-dist_to_msh**2)/dist_msh_to_msp**2)
    BL['depth'+description] = depth
    plt.figure()
    hist = plt.hist(depth[np.logical_and(depth<3,depth>-2)], bins=100)

    """
    new_BL = pd.DataFrame()
    for sat in np.unique(BL.sat.values):
        temp = BL[BL.sat.values==sat]
        depth_series = pd.Series(temp['depth'+description].values)
        uniform_depth = depth_series.rank(method="first") / (len(depth_series.dropna()) + 1)
        temp['uniform_depth'+description]=uniform_depth.values
        new_BL = pd.concat([new_BL, temp])
    new_BL = new_BL.sort_index()
    #new_BL.to_pickle(path)
    """
    
    depth_series = pd.Series(BL['depth'+description].values)
    uniform_depth = depth_series.rank(method="first") / (len(depth_series.dropna()) + 1)
    BL['uniform_depth'+description]=uniform_depth.values
    
    return BL

    
def uniformize_locally(BL, feature):
    thetas = np.linspace(np.nanmin(BL.theta.values), np.nanmax(BL.theta.values), 10)
    phis = np.linspace(np.nanmin(BL.phi.values), np.nanmax(BL.phi.values), 10)
    
    BL['index_nb'] = np.arange(len(BL))
    BL['locally_uniformized_'+feature] = np.nan
    for i in range(len(thetas)-1):
        temp = make_slice(BL,'theta',thetas[i],thetas[i+1])
        for j in range(len(phis)-1):
            temp2 = make_slice(temp,'phi',phis[j],phis[j+1])
            if len(temp2)<1000:
                raise Exception(f'Trying to uniformize {feature} over a solid angle that contains only {len(temp2)} points!')
            depths = temp2[feature].values
            depth_series = pd.Series(depths[~np.isnan(depths)])
            uniform_depth = depth_series.rank(method="first") / (len(depth_series.dropna()) + 1)
            depths[~np.isnan(depths)] = uniform_depth
            BL.iloc[temp2.index_nb.values.astype(int),-1]=depths
    return BL

In [ ]:
# [from ReorderingTemporalData/DepthComputation.ipynb, cell 14]
from matplotlib.colors import LogNorm

def diagnostic(df, name_depth):   # In order to be fast, only binned_statistic
    temp = df[['Y','Z','Vz','Vz_MSH','omni_CLA','tilt',name_depth]].dropna()
    temp['dVz']=temp.Vz.values-temp.Vz_MSH.values

    # Check spatial distribution
    fig, ax = plt.subplots()
    stat,xbins,ybins,im = binned_statistic_2d(temp.Y.values,temp.Z.values,temp[name_depth].values,statistic='median',bins=50)
    im = ax.pcolormesh(xbins,ybins,gaussian_filter_nan_datas(stat.T,1), vmin=0, vmax=1)
    fig.colorbar(im, ax=ax)
    ax.axvline(0,linestyle='--',color='k')
    ax.axhline(0,linestyle='--',color='k')
    ax.set_ylim(-17,17)
    ax.set_xlim(-17,17)
    ax.set_aspect('equal')
    fig.show()
    
    # Speed profile (not map)
    fig, ax = plt.subplots()
    temp2 = temp[abs(temp.Vz_MSH.values) > 10]
    #temp2 = temp2[abs(temp2.Vz.values) < 200]
    _,_,_,im = ax.hist2d(temp2[name_depth].values, temp2.Vz.values/temp2.Vz_MSH.values, cmin=1, bins=(np.linspace(0,1,50),np.linspace(-5,5,300)))
    ax.axhline(1,linestyle='--',color='r')
    fig.colorbar(im,ax=ax)
    ax.set_xlabel(name_depth)
    ax.set_ylabel(r'$\frac{V_z}{V_{z,MSH}}$')
    ax.set_ylim(-5,5)
    fig.show()

    """
    fig, ax = plt.subplots()
    temp2 = temp2[temp2[name_depth].values > 0.75]
    _,_,_,im = ax.hist2d(temp2[name_depth].values, temp2.Vz.values/temp2.Vz_MSH.values, cmin=100,bins=(np.linspace(0,1,50),np.linspace(-5,5,300)), norm=LogNorm())
    ax.axhline(1,linestyle='--',color='r')
    fig.colorbar(im,ax=ax)
    ax.set_xlabel(name_depth)
    ax.set_ylabel(r'$\frac{V_z}{V_{z,MSH}}$')
    ax.set_ylim(-5,5)
    fig.show()
    """

    fig, ax = plt.subplots()
    temp2 = temp[abs(temp.Vz_MSH.values) > 10]
    _,_,_,im = ax.hist2d(temp2[name_depth].values, temp2.Vz.values-temp2.Vz_MSH.values, cmin=1, bins=200)
    #ax.axhline(1,linestyle='--',color='r')
    fig.colorbar(im,ax=ax)
    ax.set_xlabel(name_depth)
    ax.set_ylabel(r'$V_z - V_{z,MSH}$')
    ax.set_ylim(-300,300)
    fig.show()
    
    # Check different CLA different depths, dVz
    nb_sectors = 5
    depths = np.linspace(0,1,nb_sectors+1)
    for min_cla, max_cla, desc in zip([-np.pi,-5*np.pi/4,np.pi/2],[-np.pi/2,-3*np.pi/4,np.pi],['southwestward IMF','southward IMF','southeastward IMF']):
        temp2 = make_slice(temp, 'omni_CLA',min_cla,max_cla)
        fig, ax = plt.subplots(ncols=nb_sectors, figsize=(nb_sectors*4,4))
        for i in range(nb_sectors):
            temp3 = make_slice(temp2,name_depth,depths[i],depths[i+1])
            stat,xbins,ybins,im = binned_statistic_2d(temp3.Y.values,temp3.Z.values,temp3.dVz.values,statistic='median',bins=50)
            stat = gaussian_filter_nan_datas(stat,3)
            vmax = np.nanmax(abs(stat))
            im = ax[i].pcolormesh(xbins,ybins,stat.T,cmap='seismic',vmin=-vmax,vmax=vmax)
            fig.colorbar(im,ax=ax[i])
            ax[i].axvline(0,linestyle='--',color='k')
            ax[i].axhline(0,linestyle='--',color='k')
            ax[i].set_xlim(-17,17)
            ax[i].set_ylim(-17,17)
            ax[i].set_aspect('equal')
            ax[i].set_title(f'{desc}\n{round(depths[i],2)} < depth < {round(depths[i+1],2)}')
        fig.show()
    
    # Check different tilt, different CLA at high depths only
    external_BL = temp[temp[name_depth].values>0.8]
    tilts = np.array([-35,-11,11,35])*np.pi/180
    min_tilt = tilts[:-1]
    max_tilt = tilts[1:]
    for min_cla, max_cla, desc in zip([-np.pi,-5*np.pi/4,np.pi/2],[-np.pi/2,-3*np.pi/4,np.pi],['southwestward IMF','southward IMF','southeastward IMF']):
        temp2 = make_slice(external_BL, 'omni_CLA',min_cla,max_cla)
        fig, ax = plt.subplots(ncols=3, figsize=(3*4,4))
        for i,(mini,maxi) in enumerate(zip(min_tilt, max_tilt)):
            temp3 = make_slice(temp2,"tilt",mini,maxi)
            stat,xbins,ybins,im = binned_statistic_2d(temp3.Y.values,temp3.Z.values,temp3.dVz.values,statistic='median',bins=50)
            stat = gaussian_filter_nan_datas(stat,3)
            vmax = np.nanmax(abs(stat))
            im = ax[i].pcolormesh(xbins,ybins,stat.T,cmap='seismic',vmin=-vmax,vmax=vmax)
            fig.colorbar(im,ax=ax[i])
            ax[i].axvline(0,linestyle='--',color='k')
            ax[i].axhline(0,linestyle='--',color='k')
            ax[i].set_xlim(-17,17)
            ax[i].set_ylim(-17,17)
            ax[i].set_aspect('equal')
            ax[i].set_title(f'{desc}, {name_depth}>0.8\n{round(mini*180/np.pi,2)} < tilt < {round(maxi*180/np.pi,2)}')
        fig.show()

In [ ]:
# [from ReorderingTemporalData/DepthComputation.ipynb, cell 32]
# Y and Z 
to_uniformize_locally = True

BL = pd.read_pickle(f'{OUT}/BL_MMS1_THEMIS_Cluster_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl')  
print(len(BL))
BL['tilt']=get_tilt(BL.index.values)
BL = compute_depth(BL,[f'spectro_{i}' for i in range(32)], 1/32*np.ones(32),'only_spectro')

BL['normalized_logNoverT']=BL.uniform_depthonly_spectro.values
r, theta, phi = mp_shue1997(BL.theta.values, BL.phi.values, coord_sys = 'spherical')
BL['r_MP_Shue'] = r
BL = reposition(BL, Rmin=BL.r_MP_Shue.values)
BL['dVz']=BL.Vz.values-BL.Vz_MSH.values

#BL = BL[abs(BL.omni_COA.values)>np.pi/4]

if to_uniformize_locally:
    BL = uniformize_locally(BL,'depthonly_spectro')
    diagnostic(BL, 'locally_uniformized_depthonly_spectro')
else:
    diagnostic(BL, 'uniform_depthonly_spectro')

In [ ]:
# [from ReorderingTemporalData/DepthComputation.ipynb, cell 35]
BL[['Np','Tp','Tperp','depthonly_spectro','locally_uniformized_depthonly_spectro','uniform_depthonly_spectro','omni_CLA','omni_COA','omni_Ma','omni_Np','omni_V','omni_B','tilt','Vx','Vy','Vz','Vz_MSH','Vx_MSH','Vy_MSH','ref_MSH_Vx','ref_MSH_Vy','ref_MSH_Vz','X','Y','Z','theta','phi','sat']].to_pickle(f'{OUT}/BL_data_for_maps_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl')